In [2]:
%pip install census us


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests

url = "https://legislature.mi.gov/Bills/Bill?ObjectName=2025-SB-0001"

# Disguise the request as a real browser to get past basic bot filtering
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

resp = requests.get(url, headers=headers, timeout=30)

# Access-check signals: status code, whether the page's unique markers are present
status = resp.status_code
has_heading = "BillHeading" in resp.text
has_content = "Jeremy Moss" in resp.text

status, len(resp.text), has_heading, has_content

/Users/seulgijung/.pyenv/versions/3.13.9/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


(200, 40396, True, True)

In [4]:
# Parse the fetched HTML and extract the three top-level fields: title, subject, primary sponsor
from bs4 import BeautifulSoup

soup = BeautifulSoup(resp.text, "html.parser")

# Title: <h1 id="BillHeading">
heading_el = soup.find(id="BillHeading")
title = heading_el.get_text(strip=True) if heading_el else None

# Subject (what the bill actually does): <div id="ObjectSubject">
subject_el = soup.find(id="ObjectSubject")
subject = subject_el.get_text(strip=True) if subject_el else None

# Primary sponsor: <a class="primarySponsor">
sponsor_el = soup.find("a", class_="primarySponsor")
primary_sponsor = sponsor_el.get_text(strip=True) if sponsor_el else None

title, primary_sponsor, subject


('Senate Bill 1 of 2025',
 'Jeremy Moss (District 7)',
 "Civil rights: public records; applicability of the freedom of information act to the legislature and governor's office; provide for. Amends sec. 2 of1976 PA 442(MCL15.232).  TIE BAR WITH:SB 0002'25")

In [5]:
# Extract multi-value fields: cosponsors (list) and action history (list of dicts)

# Cosponsors: all <a> inside #SponsorList except the primary sponsor
sponsor_links = soup.select("#SponsorList li a")
cosponsors = [
    a.get_text(strip=True)
    for a in sponsor_links
    if "primarySponsor" not in (a.get("class") or [])
]

# Actions: each <tr> in the History table has 3 <td>s (date, journal, description)
action_rows = soup.select("#History table tbody tr")
actions = []
for row in action_rows:
    cells = row.find_all("td")
    if len(cells) >= 3:
        actions.append({
            "date": cells[0].get_text(strip=True),
            "description": cells[2].get_text(strip=True),
        })

cosponsors, len(actions), actions[:3]

(['Edward W. McBroom (District 38)',
  'Stephanie Chang (District 3)',
  'John Cherry (District 27)'],
 11,
 [{'date': '1/08/2025', 'description': 'INTRODUCED BY SENATOR JEREMY MOSS'},
  {'date': '1/08/2025', 'description': 'RULES SUSPENDED'},
  {'date': '1/08/2025', 'description': 'REFERRED TO COMMITTEE OF THE WHOLE'}])

In [6]:
# Find the URL of the bill's full text (the "Introduced Bill" HTML document)
from urllib.parse import urljoin

BASE = "https://legislature.mi.gov"

doc_rows = soup.select(".billDocuments .billDocRow")

text_url = None
for row in doc_rows:
    label_el = row.select_one(".text")
    label = label_el.get_text(strip=True) if label_el else ""
    html_link = row.select_one(".html a")
    if html_link and "Introduced Bill" in label:
        text_url = urljoin(BASE, html_link["href"])
        break

# Fallback: if no "Introduced Bill" row matched, take the first available html link
if text_url is None and doc_rows:
    first_link = doc_rows[0].select_one(".html a")
    if first_link:
        text_url = urljoin(BASE, first_link["href"])

text_url

'https://legislature.mi.gov/documents/2025-2026/billintroduced/Senate/htm/2025-SIB-0001.htm'

In [7]:
# Fetch the full bill text from text_url, then build content_hash from all actions
import hashlib

# Fetch and extract the full text of the bill document
text_resp = requests.get(text_url, headers=headers, timeout=30)
text_soup = BeautifulSoup(text_resp.text, "html.parser")
full_text = text_soup.get_text(separator="\n", strip=True)

# Build content_hash by joining ALL actions (order-independent change detection)
actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
content_hash = hashlib.md5(actions_joined.encode()).hexdigest()

len(full_text), content_hash, full_text[:300]

(6745,
 '4a60eb89e18b6610f90a019b8a8d639a',
 'SENATE BILL NO. 1\nA bill to amend 1976 PA 442, entitled\n"Freedom of information act,"\nby amending section 2 (MCL 15.232), as amended by 2018\r\nPA 68.\nthe people of the state of michigan enact:\nSec. 2. As used in this act:\n(a)\r\n"Cybersecurity assessment" means an investigation undertaken by a\r\nperson,')

In [8]:
# Define scrape_bill(): fetch + parse one bill into a single dictionary
from datetime import datetime

def scrape_bill(bill_id):
    # Build the detail-page URL from the bill_id and fetch it
    detail_url = f"{BASE}/Bills/Bill?ObjectName={bill_id}"
    resp = requests.get(detail_url, headers=headers, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Chamber is derivable from the bill_id (e.g. "2025-SB-0001" -> Senate)
    chamber = "Senate" if "-SB-" in bill_id else "House"

    # Top-level single-value fields
    heading_el = soup.find(id="BillHeading")
    title = heading_el.get_text(strip=True) if heading_el else None

    subject_el = soup.find(id="ObjectSubject")
    subject = subject_el.get_text(strip=True) if subject_el else None

    sponsor_el = soup.find("a", class_="primarySponsor")
    primary_sponsor = sponsor_el.get_text(strip=True) if sponsor_el else None

    # Cosponsors: all sponsor links except the primary one
    sponsor_links = soup.select("#SponsorList li a")
    cosponsors = [
        a.get_text(strip=True)
        for a in sponsor_links
        if "primarySponsor" not in (a.get("class") or [])
    ]

    # Action history: date + description from each table row
    actions = []
    for row in soup.select("#History table tbody tr"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            actions.append({
                "date": cells[0].get_text(strip=True),
                "description": cells[2].get_text(strip=True),
            })

    # Locate the "Introduced Bill" HTML document link for the full text
    text_url = None
    doc_rows = soup.select(".billDocuments .billDocRow")
    for r in doc_rows:
        label_el = r.select_one(".text")
        label = label_el.get_text(strip=True) if label_el else ""
        link = r.select_one(".html a")
        if link and "Introduced Bill" in label:
            text_url = urljoin(BASE, link["href"])
            break
    if text_url is None and doc_rows:
        first_link = doc_rows[0].select_one(".html a")
        if first_link:
            text_url = urljoin(BASE, first_link["href"])

    # Fetch the full text if a document URL was found
    full_text = None
    if text_url:
        tr = requests.get(text_url, headers=headers, timeout=30)
        full_text = BeautifulSoup(tr.text, "html.parser").get_text(separator="\n", strip=True)

    # content_hash from all actions joined (order-independent change detection)
    actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
    content_hash = hashlib.md5(actions_joined.encode()).hexdigest()

    # Assemble the single bill record
    return {
        "bill_id": bill_id,
        "chamber": chamber,
        "url": detail_url,
        "content_hash": content_hash,
        "last_scraped": datetime.now().isoformat(timespec="seconds"),
        "title": title,
        "subject": subject,
        "primary_sponsor": primary_sponsor,
        "cosponsors": cosponsors,
        "actions": actions,
        "text_url": text_url,
        "full_text": full_text,
    }

In [13]:
# Test scrape_bill() on a single known bill
record = scrape_bill("2025-SB-0001")
{k: record[k] for k in ["bill_id", "chamber", "content_hash", "title", "primary_sponsor"]} , len(record["cosponsors"]), len(record["actions"]), len(record["full_text"])

TypeError: object of type 'NoneType' has no len()

In [14]:
# Diagnose why full_text is None: check text_url and the raw text response separately
print("text_url:", record["text_url"])

if record["text_url"]:
    diag = requests.get(record["text_url"], headers=headers, timeout=30)
    print("status:", diag.status_code)
    print("length:", len(diag.text))
    print("preview:", diag.text[:200])

text_url: None


In [12]:
# Diagnose: re-fetch the detail page fresh and check whether the Documents section exists in it
detail_url = f"{BASE}/Bills/Bill?ObjectName=2025-SB-0001"
d = requests.get(detail_url, headers=headers, timeout=30)

print("status:", d.status_code)
print("length:", len(d.text))
print("has billDocuments?:", "billDocuments" in d.text)
print("has BillHeading?:", "BillHeading" in d.text)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [15]:
# Define fetch(): a single request helper with polite delay + retry to survive bot blocking
import time

def fetch(url, delay=2.0, retries=3, backoff=5.0):
    # Polite pause before every request to reduce load and avoid triggering blocks
    time.sleep(delay)
    last_err = None
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=headers, timeout=60)
            resp.raise_for_status()  # treat 4xx/5xx as errors so they get retried
            return resp
        except Exception as e:
            last_err = e
            # Wait longer on each failed attempt before retrying
            time.sleep(backoff * (attempt + 1))
    # All retries exhausted: re-raise so the caller can log it
    raise last_err

In [16]:
# Test fetch() on the detail page and confirm the polite delay is applied
start = time.time()
resp = fetch("https://legislature.mi.gov/Bills/Bill?ObjectName=2025-SB-0001")
elapsed = time.time() - start

resp.status_code, len(resp.text), round(elapsed, 1)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [17]:
# Redefine fetch() with a reused Session, longer randomized delay, and retries
import random

# One Session reused across all requests: keeps connection/cookies, looks less bot-like
session = requests.Session()
session.headers.update(headers)

def fetch(url, min_delay=5.0, max_delay=7.0, retries=3, backoff=10.0):
    # Randomized polite pause before each request to avoid mechanical request patterns
    time.sleep(random.uniform(min_delay, max_delay))
    last_err = None
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=60)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_err = e
            # Longer wait on each failed attempt before retrying
            time.sleep(backoff * (attempt + 1))
    raise last_err

In [18]:
# Single cautious test of the hardened fetch()
resp = fetch("https://legislature.mi.gov/Bills/Bill?ObjectName=2025-SB-0001")
resp.status_code, len(resp.text), "BillHeading" in resp.text

(200, 40396, True)

In [19]:
# Redefine scrape_bill() to route every request through the hardened fetch()
def scrape_bill(bill_id):
    # Build the detail-page URL and fetch it via the hardened fetch()
    detail_url = f"{BASE}/Bills/Bill?ObjectName={bill_id}"
    resp = fetch(detail_url)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Chamber is derivable from the bill_id (e.g. "2025-SB-0001" -> Senate)
    chamber = "Senate" if "-SB-" in bill_id else "House"

    # Top-level single-value fields
    heading_el = soup.find(id="BillHeading")
    title = heading_el.get_text(strip=True) if heading_el else None

    subject_el = soup.find(id="ObjectSubject")
    subject = subject_el.get_text(strip=True) if subject_el else None

    sponsor_el = soup.find("a", class_="primarySponsor")
    primary_sponsor = sponsor_el.get_text(strip=True) if sponsor_el else None

    # Cosponsors: all sponsor links except the primary one
    sponsor_links = soup.select("#SponsorList li a")
    cosponsors = [
        a.get_text(strip=True)
        for a in sponsor_links
        if "primarySponsor" not in (a.get("class") or [])
    ]

    # Action history: date + description from each table row
    actions = []
    for row in soup.select("#History table tbody tr"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            actions.append({
                "date": cells[0].get_text(strip=True),
                "description": cells[2].get_text(strip=True),
            })

    # Locate the "Introduced Bill" HTML document link for the full text
    text_url = None
    doc_rows = soup.select(".billDocuments .billDocRow")
    for r in doc_rows:
        label_el = r.select_one(".text")
        label = label_el.get_text(strip=True) if label_el else ""
        link = r.select_one(".html a")
        if link and "Introduced Bill" in label:
            text_url = urljoin(BASE, link["href"])
            break
    if text_url is None and doc_rows:
        first_link = doc_rows[0].select_one(".html a")
        if first_link:
            text_url = urljoin(BASE, first_link["href"])

    # Fetch the full text via fetch() if a document URL was found
    full_text = None
    if text_url:
        tr = fetch(text_url)
        full_text = BeautifulSoup(tr.text, "html.parser").get_text(separator="\n", strip=True)

    # content_hash from all actions joined (order-independent change detection)
    actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
    content_hash = hashlib.md5(actions_joined.encode()).hexdigest()

    # Assemble the single bill record
    return {
        "bill_id": bill_id,
        "chamber": chamber,
        "url": detail_url,
        "content_hash": content_hash,
        "last_scraped": datetime.now().isoformat(timespec="seconds"),
        "title": title,
        "subject": subject,
        "primary_sponsor": primary_sponsor,
        "cosponsors": cosponsors,
        "actions": actions,
        "text_url": text_url,
        "full_text": full_text,
    }

In [20]:
# Test the fetch-powered scrape_bill() on a single bill
record = scrape_bill("2025-SB-0001")
(
    record["bill_id"],
    record["content_hash"],
    record["primary_sponsor"],
    len(record["cosponsors"]),
    len(record["actions"]),
    len(record["full_text"]) if record["full_text"] else None,
)

('2025-SB-0001',
 '4a60eb89e18b6610f90a019b8a8d639a',
 'Jeremy Moss (District 7)',
 3,
 11,
 6745)

In [21]:
# Fetch the Senate bill list page and check whether all bill_ids are present statically
import re

list_url = "https://legislature.mi.gov/Search/ExecuteSearch?sessions=2025-2026&docTypes=Senate%20Bill"
resp = fetch(list_url)

# Find every detail-page reference like ObjectName=2025-SB-0001 in the raw HTML
found = re.findall(r"ObjectName=(20\d\d-SB-\d+)", resp.text)
unique_ids = sorted(set(found))

len(resp.text), len(found), len(unique_ids), unique_ids[:5], unique_ids[-5:]

(762565, 0, 0, [], [])

In [22]:
# Diagnose how bill links/ids are actually represented in the 762KB search response
text = resp.text

# Count several candidate patterns to see which representation the page uses
candidates = {
    "ObjectName= (any)": len(re.findall(r"ObjectName=", text)),
    "objectName= (lower o)": len(re.findall(r"objectName=", text)),
    "/Bills/Bill": len(re.findall(r"/Bills/Bill", text)),
    "SB followed by digits": len(re.findall(r"SB[\s-]?\d+", text)),
    "GetObject": len(re.findall(r"GetObject", text)),
}

# Grab the HTML around the first occurrence of "SB" to see the actual markup
idx = text.find("SB")
snippet = text[idx-200:idx+200] if idx != -1 else "no 'SB' found"

candidates, snippet

({'ObjectName= (any)': 0,
  'objectName= (lower o)': 1141,
  '/Bills/Bill': 0,
  'SB followed by digits': 2794,
  'GetObject': 1141},
 'r>\r\n                </thead>\r\n                <tbody>\r\n                            <tr>\r\n\r\n                                        <td style="min-width:15ex;"><a href="/Home/GetObject?objectName=2025-SB-0001">SB 0001 of 2025</a></td>\r\n                                <td style="min-width:12ex;">Senate Bill</td>\r\n                                <td>\r\n                                    Civil rights:')

In [23]:
# Extract all Senate bill_ids using the correct lowercase objectName= pattern
found = re.findall(r"objectName=(20\d\d-SB-\d+)", resp.text)
senate_ids = sorted(set(found))

len(found), len(senate_ids), senate_ids[:5], senate_ids[-5:]

(1141,
 1141,
 ['2025-SB-0001',
  '2025-SB-0002',
  '2025-SB-0003',
  '2025-SB-0004',
  '2025-SB-0005'],
 ['2026-SB-1137',
  '2026-SB-1138',
  '2026-SB-1139',
  '2026-SB-1140',
  '2026-SB-1141'])

In [24]:
# Define collect_bill_ids(): gather all Senate + House bill_ids from the search list pages
def collect_bill_ids():
    # Each chamber differs only by the docTypes query value and the id code (SB/HB)
    chambers = [
        {"doc_type": "Senate%20Bill", "code": "SB"},
        {"doc_type": "House%20Bill", "code": "HB"},
    ]

    all_ids = []
    for ch in chambers:
        url = f"{BASE}/Search/ExecuteSearch?sessions=2025-2026&docTypes={ch['doc_type']}"
        r = fetch(url)
        # Extract ids for this chamber's code only (e.g. 2025-SB-0001)
        ids = re.findall(rf"objectName=(20\d\d-{ch['code']}-\d+)", r.text)
        all_ids.extend(sorted(set(ids)))

    return all_ids

In [25]:
# Run the collector and check totals for both chambers
bill_ids = collect_bill_ids()
senate = [b for b in bill_ids if "-SB-" in b]
house = [b for b in bill_ids if "-HB-" in b]

len(bill_ids), len(senate), len(house), house[:3], house[-3:]

(3371,
 1141,
 2230,
 ['2025-HB-4001', '2025-HB-4002', '2025-HB-4003'],
 ['2026-HB-6228', '2026-HB-6229', '2026-HB-6230'])

### Narrowing down to certain subjects
using time to avoid ban, it is estimated to take more than 11 hours to scrap all the bills in both the house and the senate.

In [26]:
# Count bills per category (2025-2026) to pick a focused, high-activity topic by data
from urllib.parse import quote

# Interest-aligned categories + a few high-activity comparators
categories = [
    "Environmental protection", "Natural resources", "Energy", "Water supply",
    "Women", "Family law",
    "Children", "Juveniles",
    "Labor", "Employment security", "Worker's compensation",
    "Health", "Education", "Crimes", "Elections",
]

category_counts = {}
for cat in categories:
    url = f"{BASE}/Search/ExecuteSearch?sessions=2025-2026&docTypes=Bills&category={quote(cat)}"
    r = fetch(url)
    ids = re.findall(r"objectName=(20\d\d-(?:SB|HB)-\d+)", r.text)
    category_counts[cat] = len(set(ids))

# Sort by count so the most active topics surface first
dict(sorted(category_counts.items(), key=lambda x: x[1], reverse=True))

{'Health': 216,
 'Education': 192,
 'Crimes': 175,
 'Labor': 114,
 'Children': 90,
 'Environmental protection': 89,
 'Natural resources': 84,
 'Elections': 81,
 'Energy': 49,
 'Water supply': 45,
 'Family law': 38,
 "Worker's compensation": 25,
 'Employment security': 22,
 'Juveniles': 10,
 'Women': 8}

### reduced to labor-related categories

In [27]:
# Redefine collect_bill_ids(): union of bill_ids across the three labor-related categories
def collect_bill_ids():
    # Labor topic defined as the union of these three categories
    categories = ["Labor", "Employment security", "Worker's compensation"]

    all_ids = set()
    for cat in categories:
        url = f"{BASE}/Search/ExecuteSearch?sessions=2025-2026&docTypes=Bills&category={quote(cat)}"
        r = fetch(url)
        ids = set(re.findall(r"objectName=(20\d\d-(?:SB|HB)-\d+)", r.text))
        all_ids |= ids  # union across categories, duplicates removed automatically

    return sorted(all_ids)

In [28]:
# Run the labor-topic collector and check the total and chamber split
bill_ids = collect_bill_ids()
senate = [b for b in bill_ids if "-SB-" in b]
house = [b for b in bill_ids if "-HB-" in b]

len(bill_ids), len(senate), len(house), bill_ids[:5]

(160,
 34,
 126,
 ['2025-HB-4001',
  '2025-HB-4002',
  '2025-HB-4017',
  '2025-HB-4030',
  '2025-HB-4040'])

In [30]:
# Define initial_scrape(): loop over all bill_ids, scrape each, log successes and failures
def initial_scrape(bill_ids):
    data = {}          # bill_id -> scraped record
    change_log = []    # what changed this run (all "added" on the initial run)
    error_log = []     # which bills failed and why

    for i, bill_id in enumerate(bill_ids, start=1):
        try:
            record = scrape_bill(bill_id)
            data[bill_id] = record
            change_log.append({"bill_id": bill_id, "change": "added"})
        except Exception as e:
            # Log the failure and keep going; one bad page must not stop the whole run
            error_log.append({"bill_id": bill_id, "error": str(e)})

        # Lightweight progress marker for this long-running loop
        print(f"[{i}/{len(bill_ids)}] {bill_id}")

    return data, change_log, error_log

In [31]:
# Smoke-test the loop on just the first 3 bills before the full 20-minute run
test_data, test_changes, test_errors = initial_scrape(bill_ids[:3])
len(test_data), len(test_changes), len(test_errors), list(test_data.keys())

[1/3] 2025-HB-4001
[2/3] 2025-HB-4002
[3/3] 2025-HB-4017


(3, 3, 0, ['2025-HB-4001', '2025-HB-4002', '2025-HB-4017'])

In [32]:
# Run the full initial scrape over all 160 bills, then save data + logs to JSON files
import json

# This runs the full ~20-minute scrape (160 bills x 2 requests each, with delays)
data, change_log, error_log = initial_scrape(bill_ids)

# Persist the core database and this run's logs
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("change_log.json", "w", encoding="utf-8") as f:
    json.dump(change_log, f, ensure_ascii=False, indent=2)

with open("error_log.json", "w", encoding="utf-8") as f:
    json.dump(error_log, f, ensure_ascii=False, indent=2)

len(data), len(error_log)

[1/160] 2025-HB-4001
[2/160] 2025-HB-4002
[3/160] 2025-HB-4017
[4/160] 2025-HB-4030
[5/160] 2025-HB-4040
[6/160] 2025-HB-4051
[7/160] 2025-HB-4061
[8/160] 2025-HB-4075
[9/160] 2025-HB-4096
[10/160] 2025-HB-4106
[11/160] 2025-HB-4129
[12/160] 2025-HB-4168
[13/160] 2025-HB-4253
[14/160] 2025-HB-4264
[15/160] 2025-HB-4288
[16/160] 2025-HB-4289
[17/160] 2025-HB-4290
[18/160] 2025-HB-4291
[19/160] 2025-HB-4292
[20/160] 2025-HB-4293
[21/160] 2025-HB-4294
[22/160] 2025-HB-4295
[23/160] 2025-HB-4296
[24/160] 2025-HB-4297
[25/160] 2025-HB-4316
[26/160] 2025-HB-4317
[27/160] 2025-HB-4318
[28/160] 2025-HB-4320
[29/160] 2025-HB-4321
[30/160] 2025-HB-4322
[31/160] 2025-HB-4323
[32/160] 2025-HB-4326
[33/160] 2025-HB-4405
[34/160] 2025-HB-4406
[35/160] 2025-HB-4409
[36/160] 2025-HB-4433
[37/160] 2025-HB-4435
[38/160] 2025-HB-4436
[39/160] 2025-HB-4437
[40/160] 2025-HB-4438
[41/160] 2025-HB-4439
[42/160] 2025-HB-4440
[43/160] 2025-HB-4441
[44/160] 2025-HB-4446
[45/160] 2025-HB-4447
[46/160] 2025-HB-44

KeyboardInterrupt: 

In [33]:
# Hardened fetch (8-12s delay) + resumable/checkpointed scrape_all + first 30-bill batch

def fetch(url, min_delay=8.0, max_delay=12.0, retries=3, backoff=10.0):
    # Longer randomized polite pause before each request to reduce bot-blocking
    time.sleep(random.uniform(min_delay, max_delay))
    last_err = None
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=60)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_err = e
            time.sleep(backoff * (attempt + 1))
    raise last_err

def load_json(path, default):
    # Read an existing JSON file, or return a default if it doesn't exist yet
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return default

def save_json(path, obj):
    # Write an object to a JSON file (used as a checkpoint after each bill)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def scrape_all(bill_ids):
    # Resume from whatever was already saved to disk
    data = load_json("data.json", {})
    error_log = load_json("error_log.json", [])

    for i, bill_id in enumerate(bill_ids, start=1):
        # Resume: skip bills already scraped
        if bill_id in data:
            print(f"[{i}/{len(bill_ids)}] {bill_id} (skip)")
            continue
        try:
            data[bill_id] = scrape_bill(bill_id)
            save_json("data.json", data)        # checkpoint after each bill
            print(f"[{i}/{len(bill_ids)}] {bill_id} (saved)")
        except Exception as e:
            error_log.append({"bill_id": bill_id, "error": str(e)})
            save_json("error_log.json", error_log)
            print(f"[{i}/{len(bill_ids)}] {bill_id} (ERROR)")

    return data, error_log

# Run the first batch of 30 bills (safe to re-run: resume + checkpoint)
data, error_log = scrape_all(bill_ids[:30])
len(data), len(error_log)

[1/30] 2025-HB-4001 (ERROR)
[2/30] 2025-HB-4002 (ERROR)
[3/30] 2025-HB-4017 (ERROR)


KeyboardInterrupt: 

### Using Proxy from here (ScrapeOps)

In [36]:
# Load the ScrapeOps API key from a .env file (kept out of git via .gitignore)
import os
from dotenv import load_dotenv

load_dotenv()
SCRAPEOPS_API_KEY = os.getenv("SCRAPEOPS_API_KEY")

# Sanity check: confirm the key loaded without printing it
SCRAPEOPS_API_KEY is not None

True

In [37]:
# Redefine fetch() to route all requests through the ScrapeOps proxy
from urllib.parse import urlencode

def fetch(url, min_delay=1.0, max_delay=2.0, retries=3, backoff=10.0):
    # Short pause now that the proxy rotates IPs for us
    time.sleep(random.uniform(min_delay, max_delay))

    # Wrap the target URL inside the ScrapeOps proxy request
    proxy_endpoint = "https://proxy.scrapeops.io/v1/"
    params = {"api_key": SCRAPEOPS_API_KEY, "url": url}

    last_err = None
    for attempt in range(retries):
        try:
            # Proxy can be slow (15-20s per request), so allow a long timeout
            resp = session.get(proxy_endpoint, params=params, timeout=120)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_err = e
            time.sleep(backoff * (attempt + 1))
    raise last_err

In [40]:
# Single test: confirm the proxy successfully fetches one bill detail page
resp = fetch(f"{BASE}/Bills/Bill?ObjectName=2025-SB-0001")
resp.status_code, len(resp.text), "BillHeading" in resp.text

(200, 40396, True)

In [41]:
# Scrape the first 40 bills via proxy (resume skips already-saved ones; checkpoint saves each)
data, error_log = scrape_all(bill_ids[:40])
len(data), len(error_log)

[1/40] 2025-HB-4001 (saved)
[2/40] 2025-HB-4002 (saved)
[3/40] 2025-HB-4017 (saved)
[4/40] 2025-HB-4030 (saved)
[5/40] 2025-HB-4040 (saved)
[6/40] 2025-HB-4051 (saved)
[7/40] 2025-HB-4061 (saved)
[8/40] 2025-HB-4075 (saved)
[9/40] 2025-HB-4096 (saved)
[10/40] 2025-HB-4106 (saved)
[11/40] 2025-HB-4129 (saved)
[12/40] 2025-HB-4168 (saved)
[13/40] 2025-HB-4253 (saved)
[14/40] 2025-HB-4264 (saved)
[15/40] 2025-HB-4288 (saved)
[16/40] 2025-HB-4289 (saved)
[17/40] 2025-HB-4290 (saved)
[18/40] 2025-HB-4291 (saved)
[19/40] 2025-HB-4292 (saved)
[20/40] 2025-HB-4293 (saved)
[21/40] 2025-HB-4294 (saved)
[22/40] 2025-HB-4295 (saved)
[23/40] 2025-HB-4296 (saved)
[24/40] 2025-HB-4297 (saved)
[25/40] 2025-HB-4316 (saved)
[26/40] 2025-HB-4317 (saved)
[27/40] 2025-HB-4318 (saved)
[28/40] 2025-HB-4320 (saved)
[29/40] 2025-HB-4321 (saved)
[30/40] 2025-HB-4322 (saved)
[31/40] 2025-HB-4323 (saved)
[32/40] 2025-HB-4326 (saved)
[33/40] 2025-HB-4405 (saved)
[34/40] 2025-HB-4406 (saved)
[35/40] 2025-HB-4409 (s

(40, 3)

In [42]:
# Continue: scrape up to bill 90 (already-saved ones are skipped)
data, error_log = scrape_all(bill_ids[:90])
len(data), len(error_log)

[1/90] 2025-HB-4001 (skip)
[2/90] 2025-HB-4002 (skip)
[3/90] 2025-HB-4017 (skip)
[4/90] 2025-HB-4030 (skip)
[5/90] 2025-HB-4040 (skip)
[6/90] 2025-HB-4051 (skip)
[7/90] 2025-HB-4061 (skip)
[8/90] 2025-HB-4075 (skip)
[9/90] 2025-HB-4096 (skip)
[10/90] 2025-HB-4106 (skip)
[11/90] 2025-HB-4129 (skip)
[12/90] 2025-HB-4168 (skip)
[13/90] 2025-HB-4253 (skip)
[14/90] 2025-HB-4264 (skip)
[15/90] 2025-HB-4288 (skip)
[16/90] 2025-HB-4289 (skip)
[17/90] 2025-HB-4290 (skip)
[18/90] 2025-HB-4291 (skip)
[19/90] 2025-HB-4292 (skip)
[20/90] 2025-HB-4293 (skip)
[21/90] 2025-HB-4294 (skip)
[22/90] 2025-HB-4295 (skip)
[23/90] 2025-HB-4296 (skip)
[24/90] 2025-HB-4297 (skip)
[25/90] 2025-HB-4316 (skip)
[26/90] 2025-HB-4317 (skip)
[27/90] 2025-HB-4318 (skip)
[28/90] 2025-HB-4320 (skip)
[29/90] 2025-HB-4321 (skip)
[30/90] 2025-HB-4322 (skip)
[31/90] 2025-HB-4323 (skip)
[32/90] 2025-HB-4326 (skip)
[33/90] 2025-HB-4405 (skip)
[34/90] 2025-HB-4406 (skip)
[35/90] 2025-HB-4409 (skip)
[36/90] 2025-HB-4433 (skip)
[

(90, 3)

In [43]:
# Continue: scrape up to bill 130 (already-saved ones are skipped)
data, error_log = scrape_all(bill_ids[:130])
len(data), len(error_log)

[1/130] 2025-HB-4001 (skip)
[2/130] 2025-HB-4002 (skip)
[3/130] 2025-HB-4017 (skip)
[4/130] 2025-HB-4030 (skip)
[5/130] 2025-HB-4040 (skip)
[6/130] 2025-HB-4051 (skip)
[7/130] 2025-HB-4061 (skip)
[8/130] 2025-HB-4075 (skip)
[9/130] 2025-HB-4096 (skip)
[10/130] 2025-HB-4106 (skip)
[11/130] 2025-HB-4129 (skip)
[12/130] 2025-HB-4168 (skip)
[13/130] 2025-HB-4253 (skip)
[14/130] 2025-HB-4264 (skip)
[15/130] 2025-HB-4288 (skip)
[16/130] 2025-HB-4289 (skip)
[17/130] 2025-HB-4290 (skip)
[18/130] 2025-HB-4291 (skip)
[19/130] 2025-HB-4292 (skip)
[20/130] 2025-HB-4293 (skip)
[21/130] 2025-HB-4294 (skip)
[22/130] 2025-HB-4295 (skip)
[23/130] 2025-HB-4296 (skip)
[24/130] 2025-HB-4297 (skip)
[25/130] 2025-HB-4316 (skip)
[26/130] 2025-HB-4317 (skip)
[27/130] 2025-HB-4318 (skip)
[28/130] 2025-HB-4320 (skip)
[29/130] 2025-HB-4321 (skip)
[30/130] 2025-HB-4322 (skip)
[31/130] 2025-HB-4323 (skip)
[32/130] 2025-HB-4326 (skip)
[33/130] 2025-HB-4405 (skip)
[34/130] 2025-HB-4406 (skip)
[35/130] 2025-HB-4409 (

(130, 3)

In [44]:
# Final batch: scrape all remaining bills (already-saved ones are skipped)
data, error_log = scrape_all(bill_ids)
len(data), len(error_log)

[1/160] 2025-HB-4001 (skip)
[2/160] 2025-HB-4002 (skip)
[3/160] 2025-HB-4017 (skip)
[4/160] 2025-HB-4030 (skip)
[5/160] 2025-HB-4040 (skip)
[6/160] 2025-HB-4051 (skip)
[7/160] 2025-HB-4061 (skip)
[8/160] 2025-HB-4075 (skip)
[9/160] 2025-HB-4096 (skip)
[10/160] 2025-HB-4106 (skip)
[11/160] 2025-HB-4129 (skip)
[12/160] 2025-HB-4168 (skip)
[13/160] 2025-HB-4253 (skip)
[14/160] 2025-HB-4264 (skip)
[15/160] 2025-HB-4288 (skip)
[16/160] 2025-HB-4289 (skip)
[17/160] 2025-HB-4290 (skip)
[18/160] 2025-HB-4291 (skip)
[19/160] 2025-HB-4292 (skip)
[20/160] 2025-HB-4293 (skip)
[21/160] 2025-HB-4294 (skip)
[22/160] 2025-HB-4295 (skip)
[23/160] 2025-HB-4296 (skip)
[24/160] 2025-HB-4297 (skip)
[25/160] 2025-HB-4316 (skip)
[26/160] 2025-HB-4317 (skip)
[27/160] 2025-HB-4318 (skip)
[28/160] 2025-HB-4320 (skip)
[29/160] 2025-HB-4321 (skip)
[30/160] 2025-HB-4322 (skip)
[31/160] 2025-HB-4323 (skip)
[32/160] 2025-HB-4326 (skip)
[33/160] 2025-HB-4405 (skip)
[34/160] 2025-HB-4406 (skip)
[35/160] 2025-HB-4409 (

(160, 3)

### check the three errors

In [45]:
# Inspect the 3 failed bills to see whether they're retryable
error_log

[{'bill_id': '2025-HB-4001',
  'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"},
 {'bill_id': '2025-HB-4002',
  'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"},
 {'bill_id': '2025-HB-4017',
  'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"}]

In [46]:
# Quality check: how many records have full_text, sponsor, actions populated
saved = load_json("data.json", {})
has_text = sum(1 for b in saved.values() if b.get("full_text"))
has_sponsor = sum(1 for b in saved.values() if b.get("primary_sponsor"))
has_actions = sum(1 for b in saved.values() if b.get("actions"))

len(saved), has_text, has_sponsor, has_actions

(160, 160, 160, 160)

In [47]:
# Clean error_log: drop entries whose bill_id was actually scraped successfully into data
saved = load_json("data.json", {})
error_log = load_json("error_log.json", [])

# Keep only errors for bills that are genuinely missing from data
real_errors = [e for e in error_log if e["bill_id"] not in saved]
save_json("error_log.json", real_errors)

len(real_errors)

0

In [48]:
# Define get_content_hash(): fetch only the detail page and compute the actions-based hash
def get_content_hash(bill_id):
    detail_url = f"{BASE}/Bills/Bill?ObjectName={bill_id}"
    resp = fetch(detail_url)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Extract actions the same way scrape_bill does
    actions = []
    for row in soup.select("#History table tbody tr"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            actions.append({
                "date": cells[0].get_text(strip=True),
                "description": cells[2].get_text(strip=True),
            })

    # Same hashing logic as scrape_bill (join all actions, order-independent)
    actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
    return hashlib.md5(actions_joined.encode()).hexdigest()

In [49]:
# Test: the freshly computed hash should match the stored hash for an unchanged bill
saved = load_json("data.json", {})
test_id = "2025-SB-0001"
saved.get(test_id, {}).get("content_hash"), get_content_hash(test_id)

HTTPError: 401 Client Error: Unauthorized for url: https://proxy.scrapeops.io/v1/?api_key=ce59df5c-e312-44f1-9cb7-49de6218b181&url=https%3A%2F%2Flegislature.mi.gov%2FBills%2FBill%3FObjectName%3D2025-SB-0001

In [50]:
# Diagnose the 401: read ScrapeOps' actual error message
r = session.get(
    "https://proxy.scrapeops.io/v1/",
    params={"api_key": SCRAPEOPS_API_KEY, "url": f"{BASE}/Bills/Bill?ObjectName=2025-SB-0001"},
    timeout=120,
)
print("status:", r.status_code)
print("body:", r.text[:500])

status: 401
body: {"Error":"The API key you sent with the request is invalid. Please include a valid API key."}


the initially issued API is still overriding; the code below make the new one overrule

In [51]:
# Reload .env and check the key prefix only (to verify the update without exposing the full key)
load_dotenv(override=True)  # override=True forces reload of changed .env values
SCRAPEOPS_API_KEY = os.getenv("SCRAPEOPS_API_KEY")
SCRAPEOPS_API_KEY[:8] if SCRAPEOPS_API_KEY else None

'9518bf19'

In [52]:
# Test: the freshly computed hash should match the stored hash for an unchanged bill
saved = load_json("data.json", {})
test_id = "2025-SB-0001"
saved.get(test_id, {}).get("content_hash"), get_content_hash(test_id)

(None, '4a60eb89e18b6610f90a019b8a8d639a')

In [53]:
# Test hash consistency on a bill that actually exists in our labor dataset
saved = load_json("data.json", {})
test_id = bill_ids[0]  # a real labor bill we already scraped
saved[test_id]["content_hash"], get_content_hash(test_id)

('30d8ce9cf7fa652f6a849d04e0995628', '30d8ce9cf7fa652f6a849d04e0995628')

In [54]:
# Define update_scrape(): detect new/modified labor bills via hash and refresh data + logs
def update_scrape():
    data = load_json("data.json", {})
    error_log = []                     # errors for this update run
    change_log = []                    # changes detected this update run
    run_ts = datetime.now().isoformat(timespec="seconds")

    # Re-fetch the current labor bill list so newly introduced bills are caught
    current_ids = collect_bill_ids()

    for i, bill_id in enumerate(current_ids, start=1):
        try:
            if bill_id not in data:
                # New bill introduced since the last run
                data[bill_id] = scrape_bill(bill_id)
                change_log.append({"bill_id": bill_id, "change": "added", "run": run_ts})
                save_json("data.json", data)
                print(f"[{i}/{len(current_ids)}] {bill_id} (added)")
            else:
                # Existing bill: compare hashes to see if anything changed
                new_hash = get_content_hash(bill_id)
                if new_hash != data[bill_id]["content_hash"]:
                    data[bill_id] = scrape_bill(bill_id)
                    change_log.append({"bill_id": bill_id, "change": "modified", "run": run_ts})
                    save_json("data.json", data)
                    print(f"[{i}/{len(current_ids)}] {bill_id} (modified)")
                else:
                    print(f"[{i}/{len(current_ids)}] {bill_id} (skip)")
        except Exception as e:
            error_log.append({"bill_id": bill_id, "error": str(e), "run": run_ts})
            save_json("error_log.json", error_log)
            print(f"[{i}/{len(current_ids)}] {bill_id} (ERROR)")

    # Save this run's change log
    save_json("change_log.json", change_log)
    return data, change_log, error_log

In [55]:
# Test version of update_scrape that takes an explicit target list (to test on a few bills)
def update_scrape_targets(target_ids):
    data = load_json("data.json", {})
    error_log = []
    change_log = []
    run_ts = datetime.now().isoformat(timespec="seconds")

    for i, bill_id in enumerate(target_ids, start=1):
        try:
            if bill_id not in data:
                data[bill_id] = scrape_bill(bill_id)
                change_log.append({"bill_id": bill_id, "change": "added", "run": run_ts})
                save_json("data.json", data)
                print(f"[{i}/{len(target_ids)}] {bill_id} (added)")
            else:
                new_hash = get_content_hash(bill_id)
                if new_hash != data[bill_id]["content_hash"]:
                    data[bill_id] = scrape_bill(bill_id)
                    change_log.append({"bill_id": bill_id, "change": "modified", "run": run_ts})
                    save_json("data.json", data)
                    print(f"[{i}/{len(target_ids)}] {bill_id} (modified)")
                else:
                    print(f"[{i}/{len(target_ids)}] {bill_id} (skip)")
        except Exception as e:
            error_log.append({"bill_id": bill_id, "error": str(e), "run": run_ts})
            print(f"[{i}/{len(target_ids)}] {bill_id} (ERROR)")

    return change_log, error_log

In [56]:
# Test the update logic on 3 already-scraped bills (all should report "skip")
changes, errors = update_scrape_targets(bill_ids[:3])
changes, errors

[1/3] 2025-HB-4001 (skip)
[2/3] 2025-HB-4002 (skip)
[3/3] 2025-HB-4017 (skip)


([], [])